**IA368FF - Aprendizado por Reforço**  
1º Semestre de 2024  
Prof. Denis Fantinato  

In [1]:
from copy import deepcopy
import numpy as np

from collections import defaultdict

# Criando o MDP

**Grid World 4x3**  
Vamos criar um Grid World 4x3 como um MDP.  

O MDP é definido por:  
                        MDP = (𝑆, 𝐴, 𝑅, ℙ, 𝛾)
com
- 𝑆: conjunto de possíveis estados.
- 𝐴: conjunto de ações.
- 𝑅 ∶ 𝑆 → ℝ: mapa de recompensa para cada estado.
- ℙ: probabilidade de transição de um estado para outro dada uma ação.
- 𝛾: fator de desconto. Um número entre 0 e 1.  
  
Neste mesmo bloco, definimos as probabilidades de transição de estados. Note que o agente tem 80\% de chance de seguir na direção da ação escolhida e 10\% de chance para cada direção perpendicular.

In [2]:
def createMDP():

    '''
    definição do ambiente
    '''

    S       = [(i,j) for i in range(1,5)
                     for j in range(1,4) if (i,j) != (2,2)]

    goals   = [(4,3), (4,2)]
    actions = ["UP", "DOWN", "LEFT", "RIGHT"]
    A       = {s : actions
               for s in S }

    R        = {s : -0.04 for s in S}
    R[(4,3)] =  1
    R[(4,2)] = -1

    P        = { (s,a) : pvals(s, a, S) for s in S for a in A[s] }

    gamma    = .9

    return (S,A,R,P,gamma)


def move(s, a, S):
    i, j = s
    if a == "UP":
        sp = (i, j+1)
    elif a == "DOWN":
        sp = (i, j-1)
    elif a == "LEFT":
        sp = (i-1, j)
    elif a == "RIGHT":
        sp = (i+1, j)
    elif a is None:
        return s

    if sp in S:
        return sp

    return s

def succ(a):
    return {"UP": "RIGHT", "DOWN": "LEFT", "RIGHT": "DOWN", "LEFT": "UP", None : None}[a]

def pred(a):
    return {"UP": "LEFT", "DOWN": "RIGHT", "RIGHT": "UP", "LEFT": "DOWN", None : None}[a]

def pvals(s, a, S):
    return [(0.8, move(s, a, S)), (0.1, move(s, succ(a), S)), (0.1, move(s, pred(a), S))]

# Estimativa Direta

Aprendizado Passivo.  
É necessário obter as sequências-amostras e depois estimar as utilidades

In [3]:
def directEst(model, s, goals, R, gamma, nextState, maxLen=100):
    trial = [(s, R[s])]
    while s not in goals:
        s = nextState(s)
        trial.append( (s, R[s]) )
        if len(trial) > maxLen:
            return None
    for i, (s, r) in enumerate(trial):
        u        = sum(r*(gamma**j)
                        for j, (si, r) in enumerate(trial[i:]))
        model[s] = (model[s][0] + u, model[s][1] + 1)
    return model

def runDirectEst(mdp, pi, nTrials):
    S, A, R, P, gamma = mdp

    model = defaultdict(lambda: (0.0, 0))
    s0    = (1,1)
    goals = [(4,3), (4,2)]

    for trials in range(nTrials):
        model = directEst(model, s0, goals, R, gamma, performAction(pi, P))
        if model is None:
            break

    return model

# Escolhe próximo estado dado uma ação
def performAction(pi, P):
    def nextState(s):
        ps     = P[(s, pi[s])]
        probs  = list(map(lambda x: x[0], ps))
        states = list(map(lambda x: x[1], ps))
        idx    = np.random.choice(len(states), p=probs)

        return states[idx]
    return nextState

mdp = createMDP()

# Algoritmo Genético

Criação do indivíduo e definição das etapas de cruzamento, mutação e seleção.

In [5]:
class Individuo:
    def __init__(self, cromossomo = None):
        self.cromossomo = cromossomo
        if cromossomo is None:
            self.cromossomo = np.random.choice(["UP", "DOWN", "LEFT", "RIGHT"], 12)

        self.pi      = {(i,j):self.cromossomo[3*i + j - 4] for i in range(1,5) for j in range(1,4)}
        self.fitness = avalia(self.pi)

def tournament(P):
    idx1, idx2 = np.random.choice(len(P), 2)
    if P[idx1].fitness > P[idx2].fitness:
        return P[idx1]
    return P[idx2]

def seleciona(P, n):
    return [tournament(P) for _ in range(n)]

def muta(p):
    cromossomo = p.cromossomo
    idx = np.random.choice(len(cromossomo))
    cromossomo[idx] = np.random.choice(["UP", "DOWN", "LEFT", "RIGHT"])
    return Individuo(cromossomo)

def cruza(p1, p2):
    cromossomo1 = p1.cromossomo
    cromossomo2 = p2.cromossomo
    idx = np.random.choice(len(cromossomo1))
    return Individuo(np.append(cromossomo1[:idx], cromossomo2[idx:]))

def cruzamento(P):
    return [cruza(seleciona(P, 1)[0], seleciona(P, 1)[0])
            for _ in range(len(P))]

def mutacao(P):
    return [muta(p) for p in P]

def melhor(P):
    fitness = [(-p.fitness, i) for i, p in enumerate(P)]
    idx = sorted(fitness)[0][1]
    return P[idx]


Avaliação de Cada indivíduo:

In [6]:
def evaluate(mdp, pi, nTrials):
    model = runDirectEst(mdp, pi, nTrials)
    if model is None:
        U = { s:-np.inf for s in mdp[0] }
    else:
        U     = {}
        for s, (v, n) in model.items():
            U[s] = v/n
    return U


def avalia(pi):
    U = evaluate(mdp, pi, 10)
    return U[(1,1)]

In [7]:
for i in range(3):
    P = np.random.choice(["UP", "DOWN", "LEFT", "RIGHT"], 12)
    pi = {(i,j):P[3*i + j - 4] for i in range(1,5) for j in range(1,4)}
    fitnes = avalia(pi)
    print(f'Individuo {i+1}: {P}')
    print(f'Política: {pi}')
    print(f'Fitness: {fitnes}\n')

Individuo 1: ['LEFT' 'DOWN' 'RIGHT' 'LEFT' 'DOWN' 'UP' 'UP' 'LEFT' 'UP' 'RIGHT' 'LEFT'
 'RIGHT']
Política: {(1, 1): np.str_('LEFT'), (1, 2): np.str_('DOWN'), (1, 3): np.str_('RIGHT'), (2, 1): np.str_('LEFT'), (2, 2): np.str_('DOWN'), (2, 3): np.str_('UP'), (3, 1): np.str_('UP'), (3, 2): np.str_('LEFT'), (3, 3): np.str_('UP'), (4, 1): np.str_('RIGHT'), (4, 2): np.str_('LEFT'), (4, 3): np.str_('RIGHT')}
Fitness: -inf

Individuo 2: ['DOWN' 'DOWN' 'UP' 'RIGHT' 'RIGHT' 'LEFT' 'DOWN' 'UP' 'DOWN' 'LEFT'
 'LEFT' 'RIGHT']
Política: {(1, 1): np.str_('DOWN'), (1, 2): np.str_('DOWN'), (1, 3): np.str_('UP'), (2, 1): np.str_('RIGHT'), (2, 2): np.str_('RIGHT'), (2, 3): np.str_('LEFT'), (3, 1): np.str_('DOWN'), (3, 2): np.str_('UP'), (3, 3): np.str_('DOWN'), (4, 1): np.str_('LEFT'), (4, 2): np.str_('LEFT'), (4, 3): np.str_('RIGHT')}
Fitness: -inf

Individuo 3: ['LEFT' 'DOWN' 'DOWN' 'UP' 'RIGHT' 'UP' 'LEFT' 'LEFT' 'LEFT' 'RIGHT'
 'DOWN' 'LEFT']
Política: {(1, 1): np.str_('LEFT'), (1, 2): np.str_('DOWN'

Função GA:  
O tamanho da população e número de gerações é definido aqui:

In [8]:
def GA():
    P = [Individuo() for _ in range(100)]
    for it in range(100):
        filhos = cruzamento(P)
        filhos = mutacao(filhos)
        P = seleciona(P + filhos, len(P))
        print(it, melhor(P).fitness)
    return melhor(P)

# Função Principal

In [9]:
def main():
    p = GA()
    print(p.pi, p.fitness)

main()

0 0.2883091272363092
1 0.2883091272363092
2 0.2883091272363092
3 0.36601551446
4 0.3817225484600001
5 0.3817225484600001
6 0.3817225484600001
7 0.36601551446
8 0.38003729000000014
9 0.3794822294000001
10 0.3794822294000001
11 0.3944452460000001
12 0.3867295100000001
13 0.3867295100000001
14 0.3867295100000001
15 0.3867295100000001
16 0.3867295100000001
17 0.3867295100000001
18 0.40188542000000005
19 0.40188542000000005
20 0.40188542000000005
21 0.40670775500000006
22 0.40670775500000006
23 0.40670775500000006
24 0.40670775500000006
25 0.40670775500000006
26 0.40188542000000005
27 0.40188542000000005
28 0.39361856000000006
29 0.3867295100000001
30 0.37295141000000004
31 0.37295141000000004
32 0.3944452460000001
33 0.3929296550000001
34 0.3973762236363637
35 0.3787382120000001
36 0.3787382120000001
37 0.3787382120000001
38 0.3787382120000001
39 0.38672951000000005
40 0.38672951000000005
41 0.38672951000000005
42 0.38672951000000005
43 0.38672951000000005
44 0.38986089636363647
45 0.38986